# Augmentation benchmark

Tune the actual training augmentation pipeline. Compare the same images before/after, inspect matched card crops, and measure CPU processing time. Use fixed seeds for comparisons; **New draw** changes the seed.

**Balanced** mixes clean and challenging inputs. **Hard** increases lighting and camera degradation. Real images are available for visual comparison only; training validation/test inputs remain unaugmented. This preview measures appearance and runtime, not detector accuracy.

**Save training preset** writes `configs/augmentation.json`. The training notebook loads this explicit file and logs its values. Restart its kernel before a new run. Preview outputs overwrite `data/augmentation_benchmark/current/`.


In [ ]:
from dataclasses import fields
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

from set_master.training.augmentation import AugmentationConfig, HARD_AUGMENTATION_CONFIG
from set_master.training.augmentation_benchmark import render_augmentation_benchmark
from set_master.paths import (
    SYNTHETIC_DATASETS_DIR, REAL_TESTSET_DIR, AUGMENTATION_PRESET_PATH,
    AUGMENTATION_BENCHMARK_DIR,
)
from set_master.visualization import display_image

presets = {
    'Balanced': AugmentationConfig(),
    'Hard': HARD_AUGMENTATION_CONFIG,
    'Saved training preset': AugmentationConfig.load(AUGMENTATION_PRESET_PATH),
}
source = widgets.Dropdown(options=[
    ('Synthetic training', SYNTHETIC_DATASETS_DIR / 'tabletop_scenes_v2_train'),
    ('Synthetic validation', SYNTHETIC_DATASETS_DIR / 'tabletop_scenes_v2_validation'),
    ('Real reference', REAL_TESTSET_DIR),
], description='Source')
image_ids = widgets.Text(value='1,400,800,1200,1600,2000', description='Image IDs', layout=widgets.Layout(width='550px'))
seed = widgets.IntText(value=42, description='Seed')
draw_count = widgets.BoundedIntText(value=1, min=1, max=4, description='Draws')
show_boxes = widgets.Checkbox(value=False, description='Draw transformed boxes')
preset = widgets.Dropdown(options=list(presets), value='Saved training preset', description='Preset')
preset_path = widgets.Text(value=str(AUGMENTATION_PRESET_PATH), description='Save path', layout=widgets.Layout(width='650px'))


In [ ]:
range_limits = {
    'perspective_scale': (0, 0.12, 0.005),
    'temperature_range': (3000, 15000, 100),
    'gamma_range': (50, 170, 5),
    'illumination_strength': (0, 0.7, 0.01),
    'shadow_strength': (0, 0.8, 0.01),
    'shadow_size': (0.05, 0.8, 0.01),
    'glare_strength': (0, 1, 0.01),
    'glare_radius_ratio': (0.03, 0.6, 0.01),
    'glare_spots': (1, 5, 1),
    'blur_kernel': (3, 15, 2),
    'downscale_range': (0.2, 1, 0.01),
    'noise_std': (0, 0.08, 0.001),
    'jpeg_quality': (20, 100, 1),
}
controls = {}
defaults = presets[preset.value]
for field in fields(defaults):
    name, value = field.name, getattr(defaults, field.name)
    common = dict(description=name.replace('_', ' '), continuous_update=False,
                  style={'description_width': '210px'}, layout=widgets.Layout(width='580px'))
    if isinstance(value, tuple):
        lower, upper, step = range_limits[name]
        kind = widgets.IntRangeSlider if isinstance(value[0], int) else widgets.FloatRangeSlider
        controls[name] = kind(value=value, min=lower, max=upper, step=step, **common)
    else:
        upper = 0.5 if name in ('brightness_limit', 'contrast_limit') else 1
        lower = 0.1 if name == 'min_visibility' else 0
        controls[name] = widgets.FloatSlider(value=value, min=lower, max=upper, step=0.01, **common)

sections = {
    'Geometry and annotation visibility': ['flip_probability', 'perspective_probability', 'perspective_scale', 'min_visibility'],
    'Exposure and white balance': ['brightness_probability', 'brightness_limit', 'contrast_limit', 'temperature_probability', 'temperature_range', 'temperature_strength', 'gamma_probability', 'gamma_range'],
    'Uneven illumination and soft shadows': ['illumination_probability', 'illumination_strength', 'shadow_probability', 'shadow_strength', 'shadow_size'],
    'Corner glare': ['glare_probability', 'glare_strength', 'glare_radius_ratio', 'glare_spots'],
    'Camera and resolution': ['blur_probability', 'blur_kernel', 'downscale_probability', 'downscale_range', 'noise_probability', 'noise_std', 'compression_probability', 'jpeg_quality'],
}
groups = widgets.Accordion(children=[widgets.VBox([controls[name] for name in names]) for names in sections.values()])
for i, title in enumerate(sections):
    groups.set_title(i, title)

def apply_preset(change):
    selected = presets[change['new']]
    for name, control in controls.items():
        control.value = getattr(selected, name)

preset.observe(apply_preset, names='value')


In [ ]:
preview_button = widgets.Button(description='Preview / benchmark', button_style='primary')
new_draw_button = widgets.Button(description='New draw')
save_button = widgets.Button(description='Save training preset')
output = widgets.Output()
status = widgets.Output()

def read_config():
    return AugmentationConfig(**{name: tuple(control.value) if isinstance(control.value, tuple) else control.value
                                for name, control in controls.items()})

def preview(_):
    with output:
        output.clear_output(wait=True)
        try:
            config = read_config()
            ids = [int(part.strip()) for part in image_ids.value.split(',') if part.strip()]
            for draw in range(draw_count.value):
                draw_seed = seed.value + draw
                result = render_augmentation_benchmark(Path(source.value), ids, config, draw_seed, show_boxes.value)
                result.save(AUGMENTATION_BENCHMARK_DIR / 'current' / f'draw_{draw}', config, draw_seed)
                print(f'Seed {draw_seed}: {result.milliseconds_per_image:.1f} ms/image; '
                      f'{result.retained_cards}/{result.original_cards} annotations retained')
                display_image(result.scenes)
                display_image(result.crops)
        except (ValueError, KeyError, FileNotFoundError) as error:
            print(error)

def new_draw(_):
    seed.value += 1
    preview(None)

def save_preset(_):
    with status:
        status.clear_output()
        try:
            read_config().save(Path(preset_path.value))
            print(f'Saved {preset_path.value}')
        except (ValueError, OSError) as error:
            print(error)

preview_button.on_click(preview)
new_draw_button.on_click(new_draw)
save_button.on_click(save_preset)
display(widgets.VBox([
    source, image_ids, widgets.HBox([seed, draw_count]), show_boxes, preset, groups,
    widgets.HBox([preview_button, new_draw_button]), preset_path, save_button, status, output,
]))


## Reference comparisons

For **Real reference**, use IDs such as `1,4,7,9,10,13`; for validation, use IDs within 1–200. Select the source and IDs explicitly.

Judge card-edge contrast, glare, shadow softness and blur in the crops. Some draws should stay clean. Cropped cards are retained only above the visibility threshold; out-of-frame corners are marked invisible. Black padding and duplicated reflected cards should not appear.
